# 🚀 Servidor de Descargas ColorBlend

Este notebook levanta un servidor FastAPI expuesto a internet mediante **ngrok** para procesar descargas de música y videos de Instagram.

### Pasos:
1. Introduce tu Token de ngrok abajo.
2. Ejecuta el entorno (Entorno de ejecución -> Ejecutar todas).
3. Copia la URL generada (`https://xxxx.ngrok-free.app`) y pégala en tu app.

In [ ]:
# @title 🛠️ Configuración e Instalación
!pip install yt-dlp fastapi uvicorn pyngrok nest-asyncio

import os
import uuid
import threading
import nest_asyncio
import time
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.responses import FileResponse
from pyngrok import ngrok
import uvicorn

# CONFIGURACIÓN
NGROK_TOKEN = "TU_TOKEN_NGROK" # @param {type:"string"}
PORT = 8000

app = FastAPI(title="ColorBlend Download Server")
nest_asyncio.apply()

def clean_up(file_path: str):
    if os.path.exists(file_path):
        try:
            os.remove(file_path)
            print(f"🗑️ Archivo temporal eliminado: {file_path}")
        except Exception as e:
            print(f"⚠️ No se pudo borrar {file_path}: {e}")

@app.get("/")
async def root():
    return {"status": "online", "message": "ColorBlend Server is running"}

@app.post("/download-instagram")
async def download_instagram(payload: dict, background_tasks: BackgroundTasks):
    url = payload.get("url")
    if not url:
        raise HTTPException(status_code=400, detail="URL missing")

    temp_id = str(uuid.uuid4())
    output_template = f"video_{temp_id}.mp4"
    
    print(f"📥 Procesando video: {url}")
    
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'outtmpl': output_template,
        'quiet': True,
        'no_warnings': True
    }

    try:
        import yt_dlp
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        if not os.path.exists(output_template):
            raise Exception("Fallo la descarga")

        background_tasks.add_task(clean_up, output_template)
        return FileResponse(path=output_template, media_type='video/mp4', filename="video.mp4")
    
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/download-music")
async def download_music(payload: dict, background_tasks: BackgroundTasks):
    url = payload.get("url")
    temp_id = str(uuid.uuid4())
    output = f"music_{temp_id}.m4a"
    ydl_opts = {'format': 'bestaudio[ext=m4a]/best', 'outtmpl': output, 'quiet': True}
    try:
        import yt_dlp
        with yt_dlp.YoutubeDL(ydl_opts) as ydl: ydl.download([url])
        background_tasks.add_task(clean_up, output)
        return FileResponse(path=output, media_type='audio/mp4', filename="music.m4a")
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# INICIAR SERVIDOR
def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
    server = uvicorn.Server(config)
    server.run()

if NGROK_TOKEN == "TU_TOKEN_NGROK":
    print("⚠️ ERROR: Debes poner tu Token de ngrok en la variable NGROK_TOKEN")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(PORT).public_url
    print(f"\n🌍 SERVIDOR ACTIVO")
    print(f"🔗 URL PÚBLICA: {public_url}")
    print("--------------------------------------------------\n")
    
    thread = threading.Thread(target=run_server, daemon=True)
    thread.start()
    
    print("✅ Servidor corriendo en segundo plano. No cierres esta celda.")
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("⏹️ Servidor detenido.")